# Qt–Ft agglomeration: analysis

Works on finished runs in `runs/single/` and `runs/ensemble/`. Nothing here changes the simulation output;
results are written to `<run>/analysis/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd
from qtft import data_path, ensemble_stats, load_stats, compare, update_index

DATA = data_path()               # <repo>/runs

## 1. All runs

In [ ]:
update_index(DATA)              # one line per run: status, key parameters, what differs from the defaults

## 2. Ensemble statistics of one run

In [ ]:
run_dir = DATA / "ensemble" / "<run name>"   # group and name from the index above

stats = ensemble_stats(run_dir) # mean ± SD over finished replicas; saved to <run>/analysis/
stats["final"]                  # final-state values (mean, SD, number of replicas)

In [ ]:
stats["per_replica"]            # half-times, percolation and final values per replica

## 3. Full report of one run

Ensemble statistics, structure analysis with OVITO (Rg, compactness, cluster centres, nearest-neighbour
distances, composition; every `every`-th trajectory frame), overlap diagnostics, the calcium protocol figure, the binding rate k_a (Collins–Kimball), all figures
(PDF + SVG + PNG, each panel also alone in `figures/<figure>/`) and the final-state table
(CSV + LaTeX). Everything goes to `<run>/analysis/`.

In [ ]:
from qtft import make_report
out = make_report(run_dir, every=10)     # every=1: all frames (slower)
pd.read_csv(out / "final_table.csv", index_col=0)

In [ ]:
from IPython.display import Image
Image(filename=out / "figures" / "overview.png", width=900)

In [ ]:
from qtft.figures import load_run, fig_calcium

# calcium protocol: calcium and loading f, rate constants, binding and unbinding rates, apparent k_a,
# bonds and broken bonds, fraction bound (dotted lines: influx start and end); cluster sizes are in the overview
fig_calcium(load_run(run_dir));

In [ ]:
from qtft import Params, binding_rate
from qtft.figures import load_run, fig_binding_rate

# association rate k_a (Collins–Kimball contact reaction) from the start of the bond curve, with the theory;
# saved to <run>/analysis/binding_rate.csv, figure in analysis/figures/binding_rate.*
# needs a calcium step (ca_rise_s = 0): the fit assumes the peak rate from the influx start on
if Params.load(run_dir / "params.json").ca_rise_s > 0:
    print("calcium rises gradually (ca_rise_s > 0): binding rate not measured")
else:
    rate = binding_rate(run_dir)
    fig_binding_rate(load_run(run_dir), rate=rate)
    display(rate["summary"])

Individual figures can also be drawn and adjusted interactively:

```python
from qtft.figures import load_run, fig_kinetics
run = load_run(run_dir)
fig_kinetics([run], traces=True)          # with individual replicas
```

In [ ]:
pd.read_csv(out / "overlap_final.csv", index_col=0)      # interpenetration per pair family (last frames)

## 4. Compare runs

Skipped on *Run All*: to run a cell, remove its first line `%%script false --no-raise-error`.

In [ ]:
%%script false --no-raise-error
from qtft import compare_report
runs = [DATA / "ensemble" / "<run A>", DATA / "ensemble" / "<run B>"]
compare(runs)                                   # table, columns labelled by the parameters that differ


In [ ]:
%%script false --no-raise-error
compare_report(runs, DATA / "comparisons" / "<name>")   # comparison figures + table (CSV, LaTeX)

## 5. Parameter sweep

Skipped on *Run All*: to run a cell, remove its first line `%%script false --no-raise-error`.

Results of a sweep built in `control.ipynb` (section 4): one row per variant with the swept parameters and
mean ± SD over the finished replicas of the final-state values (saved to `<sweep>/sweep_results.csv`), a plot of
one result against the first swept parameter (one line per value of the others), and the full side-by-side
comparison of all runs.

In [ ]:
%%script false --no-raise-error
from qtft import sweep_results, sweep_plot, sweep_runs

sweep = DATA / "sweeps" / "<name>"
results = sweep_results(sweep)
results.round(3)

In [ ]:
%%script false --no-raise-error
sweep_plot(results, y="final_frac_bound");    # y: a result column, e.g. t_half_bonds_ms; x=...: another swept parameter

In [ ]:
%%script false --no-raise-error
compare(sweep_runs(sweep))

## 6. FIB-SEM export

Skipped on *Run All*: to run a cell, remove its first line `%%script false --no-raise-error`.

Final frame of every finished replica in the schema of the FIB-SEM segmentation pipeline:
encapsulins only, cluster IDs from the bond graph, clusters unwrapped across the periodic box.
Written to `<run>/rep_XX/fibsem/` (CSV, metadata JSON, x–y check plot).

In [ ]:
%%script false --no-raise-error
from qtft import fibsem_export_run
folders = fibsem_export_run(run_dir, voxel_nm=4.0)
pd.read_csv(folders[0] / "sim_encapsulin_centroids_clustered.csv").head()